## 1. Importaciones y configuración

Se importan las librerías necesarias para procesar las imágenes
y representar gráficamente las medidas geométricas.

Se reutilizan las funciones de segmentar.py y medidas.py,
evitando duplicar el código de la vía clásica.

Inicialmente se utilizan imágenes sintéticas para comprobar
el funcionamiento del cuaderno. Los umbrales definitivos se
obtendrán exclusivamente con el dataset real de entrenamiento.

In [ ]:
from pathlib import Path
import sys

import cv2
import numpy as np
import matplotlib.pyplot as plt


# Localizar la raíz del paquete signos_gestos.
RAIZ_PAQUETE = Path.cwd()

if RAIZ_PAQUETE.name == "herramientas":
    RAIZ_PAQUETE = RAIZ_PAQUETE.parent

sys.path.insert(0, str(RAIZ_PAQUETE))

from signos_gestos.clasico.segmentar import mascara_mano
from signos_gestos.clasico.medidas import medidas


# Directorio provisional de imágenes sintéticas.
CARPETA_DATOS = Path("/tmp/pruebas_gestos_sinteticas/bgr")

# Letras que reconoce la vía clásica.
LETRAS = ["A", "B", "L", "O", "V", "W"]

print("Directorio de datos:", CARPETA_DATOS)
print("Directorio disponible:", CARPETA_DATOS.is_dir())

## 2. Carga del dataset

Se localizan las imágenes disponibles y se asigna a cada una
su letra correspondiente a partir del nombre del archivo.

Inicialmente se utilizan imágenes sintéticas para comprobar
el funcionamiento del procesamiento.

Solo se utilizarán imágenes de entrenamiento para analizar
las medidas y establecer los umbrales de clasificación.

In [ ]:
# Lista de imágenes y sus letras correspondientes.
imagenes = []

for archivo in sorted(CARPETA_DATOS.glob("*.png")):

    letra = archivo.stem

    # Seleccionar únicamente las seis letras de la vía clásica.
    if letra not in LETRAS:
        continue

    imagenes.append({
        "ruta": archivo,
        "letra": letra
    })


# Mostrar el número de imágenes disponibles por letra.
for letra in LETRAS:
    cantidad = sum(1 for imagen in imagenes if imagen["letra"] == letra)
    print(f"{letra}: {cantidad} imágenes")

print(f"\nTotal: {len(imagenes)} imágenes")

## 3. Segmentación y extracción de medidas

Para cada imagen del conjunto de entrenamiento se obtiene
la máscara binaria mediante mascara_mano().

Posteriormente, se calculan las medidas geométricas utilizando
la función medidas().

Se almacenan los resultados junto con la letra correspondiente
para poder analizarlos y representarlos gráficamente.

In [ ]:
# Lista donde se almacenan las medidas de cada imagen.
resultados = []

for imagen in imagenes:

    # Cargar la imagen en formato BGR.
    frame = cv2.imread(str(imagen["ruta"]))

    if frame is None:
        print(f"Error al cargar: {imagen['ruta']}")
        continue

    # Segmentar la mano.
    mascara = mascara_mano(frame)

    # Calcular las medidas geométricas.
    resultado = medidas(mascara)

    # Añadir la letra y el nombre del archivo.
    resultado["letra"] = imagen["letra"]
    resultado["archivo"] = imagen["ruta"].name

    resultados.append(resultado)


# Comprobar los resultados obtenidos.
validas = sum(1 for r in resultados if r["valida"])

print(f"Imágenes procesadas: {len(resultados)}")
print(f"Máscaras válidas: {validas}")
print(f"Máscaras no válidas: {len(resultados) - validas}")

## 4. Tabla de medidas geométricas

Se muestran las medidas obtenidas para cada imagen:

- Área del contorno.
- Presencia de agujeros.
- Solidez.
- Alargamiento.
- Número de dedos separados.
- Ángulo entre dedos.

Esta primera comparación permite identificar qué características
pueden ser útiles para diferenciar las letras.

In [ ]:
# Seleccionar únicamente las máscaras válidas para el análisis geométrico.
resultados_validos = [r for r in resultados if r["valida"]]

# Mostrar la cabecera de la tabla.
print(f"{'Letra':<8} {'Área':>10} {'Agujero':>10} "
      f"{'Solidez':>10} {'Alargamiento':>14} "
      f"{'Dedos':>8} {'Ángulo':>10}")

print("-" * 86)

# Mostrar las medidas de cada imagen.
for r in resultados_validos:

    agujero = "Sí" if r["tiene_agujero"] else "No"

    angulo = r["angulo_dedos"]

    if angulo is None:
        angulo = "-"
    else:
        angulo = f"{angulo:.2f}"

    print(f"{r['letra']:<8} "
          f"{r['area']:>10.1f} "
          f"{agujero:>10} "
          f"{r['solidez']:>10.3f} "
          f"{r['alargamiento']:>14.3f} "
          f"{r['n_dedos']:>8} "
          f"{angulo:>10}")

print(f"\nMáscaras no válidas: {len(resultados) - len(resultados_validos)}")

## 5. Análisis de agujeros interiores

Se representa la presencia de agujeros interiores para cada letra.

Esta característica permite identificar la letra O, ya que la unión
del dedo índice y el pulgar forma un hueco cerrado en la silueta.

Se espera que el resto de las letras no presenten agujeros interiores
significativos.

In [ ]:
# Asignar un color a cada letra para todas las gráficas.
COLORES = {
    "A": "tab:blue",
    "B": "tab:orange",
    "L": "tab:green",
    "O": "tab:red",
    "V": "tab:purple",
    "W": "tab:brown"
}

plt.figure(figsize=(8, 4))

for letra in LETRAS:

    # Obtener la presencia de agujeros para cada imagen de la letra.
    valores = [
        int(r["tiene_agujero"])
        for r in resultados_validos
        if r["letra"] == letra
    ]

    # Representar las medidas de la letra.
    plt.scatter(
        [letra] * len(valores),
        valores,
        color=COLORES[letra],
        s=80,
        alpha=0.7
    )

plt.title("Presencia de agujeros interiores por letra")
plt.xlabel("Letra")
plt.ylabel("Presencia de agujero")
plt.yticks([0, 1], ["No", "Sí"])
plt.ylim(-0.2, 1.2)
plt.grid(axis="y", alpha=0.3)
plt.show()

## 6. Análisis del número de dedos separados

Se representa el número de dedos separados detectados mediante
los defectos de convexidad del contorno.

Esta medida permite establecer tres grupos de letras:

- A y B: 0 dedos separados.
- L y V: 2 dedos separados.
- W: 3 dedos separados.

La letra O se identifica previamente mediante su agujero interior.

In [ ]:
plt.figure(figsize=(8, 4))

for letra in LETRAS:

    # Obtener el número de dedos separados de cada imagen.
    valores = [
        r["n_dedos"]
        for r in resultados_validos
        if r["letra"] == letra
    ]

    # Representar las medidas de cada letra.
    plt.scatter(
        [letra] * len(valores),
        valores,
        color=COLORES[letra],
        s=80,
        alpha=0.7
    )

plt.title("Número de dedos separados por letra")
plt.xlabel("Letra")
plt.ylabel("Número de dedos separados")
plt.yticks(range(6))
plt.grid(axis="y", alpha=0.3)
plt.show()

## 7. Análisis del alargamiento

Se representa el alargamiento de la silueta de cada letra.

Esta medida se calcula dividiendo el lado mayor entre el lado
menor del rectángulo de área mínima que contiene la mano.

Permite diferenciar principalmente la A (puño cerrado y compacto)
de la B (dedos juntos y extendidos).

Los resultados permitirán establecer un umbral de separación
entre ambas letras.

In [ ]:
plt.figure(figsize=(8, 4))

for letra in LETRAS:

    # Obtener el alargamiento de cada imagen.
    valores = [
        r["alargamiento"]
        for r in resultados_validos
        if r["letra"] == letra
    ]

    # Representar las medidas de cada letra.
    plt.scatter(
        [letra] * len(valores),
        valores,
        color=COLORES[letra],
        s=80,
        alpha=0.7
    )

plt.title("Alargamiento por letra")
plt.xlabel("Letra")
plt.ylabel("Alargamiento")
plt.grid(axis="y", alpha=0.3)
plt.show()

## 8. Análisis de la solidez

Se representa la solidez de la silueta de cada letra.

Esta medida se calcula dividiendo el área del contorno exterior
entre el área de su envolvente convexa.

Las letras con formas compactas presentan valores próximos a 1,
mientras que las letras con separaciones entre los dedos
presentan valores inferiores.

Esta característica puede ayudar a diferenciar grupos de letras
y descartar gestos desconocidos.

In [ ]:
plt.figure(figsize=(8, 4))

for letra in LETRAS:

    # Obtener la solidez de cada imagen.
    valores = [
        r["solidez"]
        for r in resultados_validos
        if r["letra"] == letra
    ]

    # Representar las medidas de cada letra.
    plt.scatter(
        [letra] * len(valores),
        valores,
        color=COLORES[letra],
        s=80,
        alpha=0.7
    )

plt.title("Solidez por letra")
plt.xlabel("Letra")
plt.ylabel("Solidez")
plt.ylim(0, 1.05)
plt.grid(axis="y", alpha=0.3)
plt.show()

## 9. Análisis del ángulo entre dedos

Se representa el ángulo formado entre los dos dedos separados
de la mano.

Esta medida permite diferenciar principalmente la letra L,
que presenta una apertura mayor entre el pulgar y el índice,
de la letra V, que presenta una apertura menor entre el índice
y el dedo corazón.

Solo se representan las imágenes en las que ha sido posible
calcular el ángulo.

In [ ]:
plt.figure(figsize=(8, 4))

for letra in LETRAS:

    # Obtener los ángulos disponibles de cada letra.
    valores = [
        r["angulo_dedos"]
        for r in resultados_validos
        if r["letra"] == letra
        and r["angulo_dedos"] is not None
    ]

    # Representar las medidas de cada letra.
    plt.scatter(
        [letra] * len(valores),
        valores,
        color=COLORES[letra],
        s=80,
        alpha=0.7
    )

plt.title("Ángulo entre dedos por letra")
plt.xlabel("Letra")
plt.ylabel("Ángulo (grados)")
plt.ylim(0, 180)
plt.grid(axis="y", alpha=0.3)
plt.show()

## 10. Ajuste de los umbrales de medidas.py

Las gráficas anteriores muestran las medidas **ya decididas** (si hay agujero, cuántos dedos).
Para elegir los umbrales de `medidas.py` hay que ver los **valores en bruto** que hay detrás de
cada decisión, incluidos los que el umbral descarta:

- `FRACCION_MIN_AGUJERO`: tamaño del agujero más grande de cada mano, respecto al área de la mano.
- `FRACCION_MIN_PROFUNDIDAD` y `ANGULO_MAX_HUECO`: profundidad y ángulo de **todos** los defectos
  de convexidad del contorno (los posibles huecos entre dedos).

En cada gráfica, la línea discontinua es el umbral actual. Si las letras quedan bien separadas
por la línea, el umbral sirve; si no, se mueve al hueco entre los grupos y se actualiza la
constante en `medidas.py`, con un comentario que diga de dónde sale el nuevo valor.

Los umbrales de `reconocer.py` (alargamiento, ángulo entre dedos y solidez) se eligen con las
gráficas de los apartados 7, 8 y 9.

In [ ]:
from signos_gestos.clasico import medidas as med

# Valores en bruto de cada imagen.
filas_agujero = []    # (letra, fracción del agujero más grande)
filas_defectos = []   # (letra, profundidad relativa, ángulo en la base)

for imagen in imagenes:

    frame = cv2.imread(str(imagen["ruta"]))
    if frame is None:
        continue

    contorno, agujeros = med.contorno_mano(mascara_mano(frame))
    if contorno is None:
        continue

    area = cv2.contourArea(contorno)
    filas_agujero.append((imagen["letra"], med.mayor_fraccion_agujero(agujeros, area)))

    for defecto in med.defectos_convexidad(contorno):
        filas_defectos.append((imagen["letra"], defecto["profundidad"], defecto["angulo"]))

print(f"Manos analizadas: {len(filas_agujero)}")
print(f"Defectos de convexidad encontrados: {len(filas_defectos)}")

### 10.1 Tamaño del agujero → FRACCION_MIN_AGUJERO

Solo la O debería quedar por encima de la línea; el resto de letras, por debajo (sus «agujeros»
son ruido de la segmentación). El umbral bueno está entre el valor más alto de las demás letras
y el más bajo de la O.

In [ ]:
plt.figure(figsize=(8, 4))

for letra in LETRAS:
    valores = [f for l, f in filas_agujero if l == letra]
    plt.scatter([letra] * len(valores), valores, color=COLORES[letra], s=80, alpha=0.7)

plt.axhline(med.FRACCION_MIN_AGUJERO, color="gray", linestyle="--",
            label=f"FRACCION_MIN_AGUJERO = {med.FRACCION_MIN_AGUJERO}")
plt.title("Agujero más grande / área de la mano")
plt.xlabel("Letra")
plt.ylabel("Fracción del área de la mano")
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.show()

# Valores clave para elegir el corte.
o = [f for l, f in filas_agujero if l == "O"]
resto = [f for l, f in filas_agujero if l != "O"]
if o:
    print(f"O: el agujero más pequeño ocupa {min(o):.4f} de la mano")
if resto:
    print(f"Resto de letras: el agujero más grande ocupa {max(resto):.4f} de la mano")

### 10.2 Defectos de convexidad → FRACCION_MIN_PROFUNDIDAD y ANGULO_MAX_HUECO

Cada punto es un defecto de convexidad de una mano. Solo cuentan como **hueco entre dedos** los
de la zona **arriba a la izquierda**: profundos (por encima de la línea horizontal) y estrechos
(a la izquierda de la línea vertical).

Huecos esperados por mano: A y B → 0, L y V → 1, W → 2 (la O se reconoce antes por el agujero).

- Si caen puntos de la A o la B en la zona de arriba a la izquierda, se cuentan dedos que no hay.
- Si los huecos reales de la L, la V o la W quedan fuera de esa zona, se pierden dedos.

En los dos casos hay que mover las líneas hasta que separen bien los grupos.

In [ ]:
plt.figure(figsize=(8, 6))

for letra in LETRAS:
    puntos = [(a, p) for l, p, a in filas_defectos if l == letra]
    if not puntos:
        continue
    angulos, profundidades = zip(*puntos)
    plt.scatter(angulos, profundidades, color=COLORES[letra], s=40, alpha=0.6, label=letra)

plt.axhline(med.FRACCION_MIN_PROFUNDIDAD, color="gray", linestyle="--")
plt.axvline(med.ANGULO_MAX_HUECO, color="gray", linestyle="--")
plt.text(5, med.FRACCION_MIN_PROFUNDIDAD + 0.02, "cuentan como hueco entre dedos", color="gray")
plt.title("Defectos de convexidad: profundidad frente a ángulo")
plt.xlabel(f"Ángulo en la base (°) · ANGULO_MAX_HUECO = {med.ANGULO_MAX_HUECO}")
plt.ylabel(f"Profundidad / radio · FRACCION_MIN_PROFUNDIDAD = {med.FRACCION_MIN_PROFUNDIDAD}")
plt.xlim(0, 180)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 11. Conclusiones y selección de umbrales

A partir de las medidas geométricas obtenidas se estudia
la separación entre las distintas letras.

Las principales características de clasificación son:

- Agujeros interiores: identificación de la O.
- Número de dedos separados: diferenciación de grupos de letras.
- Alargamiento: diferenciación entre A y B.
- Ángulo entre dedos: diferenciación entre L y V.
- Solidez: comprobación adicional y rechazo de gestos desconocidos.

Se calculan los valores mínimos, máximos y medios de las medidas
para cada letra, con el objetivo de facilitar la selección
de los umbrales del clasificador.

Los umbrales definitivos se establecerán únicamente a partir
del conjunto real de entrenamiento.

In [ ]:
MEDIDAS_ANALISIS = [
    "tiene_agujero",
    "n_dedos",
    "alargamiento",
    "solidez",
    "angulo_dedos"
]

for medida in MEDIDAS_ANALISIS:

    print(f"\n--- {medida.upper()} ---")

    for letra in LETRAS:

        # Obtener los valores disponibles de cada medida y letra.
        valores = [
            float(r[medida])
            for r in resultados_validos
            if r["letra"] == letra
            and r[medida] is not None
        ]

        if len(valores) == 0:
            continue

        print(
            f"{letra}: "
            f"Min={np.min(valores):.3f} | "
            f"Max={np.max(valores):.3f} | "
            f"Media={np.mean(valores):.3f} | "
            f"N={len(valores)}"
        )

### Estrategia de clasificación propuesta

El clasificador de visión clásica seguirá una estructura
de decisión jerárquica:

1. Comprobar que la máscara contiene un contorno válido.
2. Comprobar la presencia de un agujero interior.
3. Analizar el número de dedos separados.
4. Diferenciar A y B mediante el alargamiento.
5. Diferenciar L y V mediante el ángulo entre dedos.
6. Identificar la W mediante sus tres dedos separados.
7. Utilizar la solidez como comprobación adicional.
8. Devolver "?" cuando no exista una clasificación clara.

Los resultados obtenidos servirán para establecer las reglas
del módulo reconocer.py y revisar, si fuera necesario,
los parámetros de segmentar.py y medidas.py.

La evaluación final se realizará con imágenes independientes
del conjunto utilizado para seleccionar los umbrales.